In [128]:
def parse_verilog_number(value, width=64):
    """
    Accepts:
      - 64'sd123
      - -64'sd123
      - 64'h1a2b
      - -64'h1a2b
      - 0x1a2b
      - plain decimal like 123
    Returns a signed Python int.
    """
    s = str(value).strip().replace("_", "")
    negative = False

    if s.startswith("-"):
        negative = True
        s = s[1:]

    low = s.lower()

    if "'sd" in low:
        number = int(low.split("'sd", 1)[1], 10)
        return -number if negative else number

    if "'sh" in low or "'h" in low:
        number = int(low.split("'h", 1)[1], 16)
        if number >= (1 << (width - 1)):
            number -= (1 << width)
        return -number if negative else number

    if low.startswith("0x"):
        number = int(low, 16)
        if number >= (1 << (width - 1)):
            number -= (1 << width)
        return -number if negative else number

    number = int(low, 10)
    return -number if negative else number
    
def parse_verilog_number(value, width=64):
    """
    Accepts:
      - 64'sd123
      - -64'sd123
      - 64'h1a2b
      - -64'h1a2b
      - 0x1a2b
      - plain decimal like 123
    Returns a signed Python int.
    """
    s = str(value).strip().replace("_", "")
    negative = False

    if s.startswith("-"):
        negative = True
        s = s[1:]

    low = s.lower()

    if "'sd" in low:
        number = int(low.split("'sd", 1)[1], 10)
        return -number if negative else number

    if "'sh" in low or "'h" in low:
        number = int(low.split("'h", 1)[1], 16)
        if number >= (1 << (width - 1)):
            number -= (1 << width)
        return -number if negative else number

    if low.startswith("0x"):
        number = int(low, 16)
        if number >= (1 << (width - 1)):
            number -= (1 << width)
        return -number if negative else number

    number = int(low, 10)
    return -number if negative else number


def signed_int_to_verilog_hex(value, width=64):
    mask = (1 << width) - 1
    unsigned_value = value & mask
    hex_digits = width // 4
    return f"{width}'h{unsigned_value:0{hex_digits}x}"


def signed64_to_hex(value):
    return signed_int_to_verilog_hex(value, width=64)


def fixed_to_float(value, frac_bits):
    return value / float(1 << frac_bits)


def q_fixed_to_float(value, frac_bits):
    return fixed_to_float(value, frac_bits)


def float_to_q_fixed(x, frac_bits, width=64):
    scaled = int(round(x * (1 << frac_bits)))
    mask = (1 << width) - 1
    scaled &= mask
    if scaled >= (1 << (width - 1)):
        scaled -= (1 << width)
    return scaled





def float_to_q_fixed_local(x, frac_bits, width=64):
    scaled = int(round(x * (1 << frac_bits)))
    mask = (1 << width) - 1
    scaled &= mask
    if scaled >= (1 << (width - 1)):
        scaled -= (1 << width)
    return scaled

def sincos_to_qhex(value_str, frac_bits=FRAC_BITS, width=WIDTH):
    signed_int = parse_verilog_number(value_str, width)
    val_float = fixed_to_float(signed_int, frac_bits)
    s = math.sin(val_float)
    c = math.cos(val_float)
    s_q = float_to_q_fixed_local(s, frac_bits, width)
    c_q = float_to_q_fixed_local(c, frac_bits, width)
    return {
        "input_float": val_float,
        "sin_float": s,
        "sin_hex": signed_int_to_verilog_hex(s_q, width),
        "cos_float": c,
        "cos_hex": signed_int_to_verilog_hex(c_q, width),
    }

In [129]:
# === Helper function to display in human-readable format ===
def display_qformat_value(value_str, frac_bits, label="Value"):
    """Parse and display a value in both hex and float."""
    signed_int = parse_verilog_number(value_str, WIDTH)
    float_val = fixed_to_float(signed_int, frac_bits)
    hex_val = signed_int_to_verilog_hex(signed_int, WIDTH)
    
    print(f"{label}:")
    print(f"  Input:        {value_str}")
    print(f"  Hex:          {hex_val}")
    print(f"  Decimal Int:  {signed_int}")
    print(f"  Float (Q{WIDTH-FRAC_BITS}.{FRAC_BITS}): {float_val:.10f}")
    return signed_int, float_val


# === Display inputs ===
print("=" * 70)
print("Q3.61 FIXED-POINT MULTIPLICATION EXAMPLE")
print("=" * 70)
print()

int_a, float_a = display_qformat_value(input_a, FRAC_BITS, "Input A")
print()
int_b, float_b = display_qformat_value(input_b, FRAC_BITS, "Input B")
print()

# === Multiply ===
result = multiply_q_fixed(input_a, input_b, width=WIDTH, frac_bits=FRAC_BITS)

print("=" * 70)
print("MULTIPLICATION RESULT")
print("=" * 70)
print(f"  {float_a:.10f} × {float_b:.10f} = {result['float']:.10f}")
print()
print(f"  Decimal Int:           {result['decimal_signed_int']}")
print(f"  Hex (64-bit):          {result['hex']}")
print(f"  Binary (2's complement): {result['binary_2s_complement']}")
print()
print(f"  Intermediate (128-bit product):")
print(f"    Product Int:         {result['product_128_signed_int']}")
print(f"    Before 64-bit wrap:  {result['result_before_64bit_wrap']}")
print(f"    Hex (128-bit):       {result['hex_128b_product']}")
print()

# === Verify with math ===
expected = float_a * float_b
print("=" * 70)
print("VERIFICATION")
print("=" * 70)
print(f"  Expected (float math):  {expected:.10f}")
print(f"  Result (Q format):      {result['float']:.10f}")
print(f"  Match: {abs(result['float'] - expected) < 1e-10}")

Q3.61 FIXED-POINT MULTIPLICATION EXAMPLE

Input A:
  Input:        0x114a280fb5068184
  Hex:          64'h114a280fb5068184
  Decimal Int:  1245852294848086404
  Float (Q3.61): 0.5403023059

Input B:
  Input:        0xE512ab70f6f321a5
  Hex:          64'he512ab70f6f321a5
  Decimal Int:  -1940299987775446619
  Float (Q3.61): -0.8414709848

MULTIPLICATION RESULT
  0.5403023059 × -0.8414709848 = -0.4546487134

  Decimal Int:           -1048348557470994948
  Hex (64-bit):          64'hf173848a9725edfc
  Binary (2's complement): 1111000101110011100001001000101010010111001001011110110111111100

  Intermediate (128-bit product):
    Product Int:         -2417327192463754166469416468601668076
    Before 64-bit wrap:  -1048348557470994948
    Hex (128-bit):       128'hfe2e709152e4bdbf8d76c3d74c2f7e14

VERIFICATION
  Expected (float math):  -0.4546487134
  Result (Q format):      -0.4546487134
  Match: True


In [130]:
import math
# comput the sin and cos and represent result in hex format into Q3.61 and Q4.60 
FRAC_BITS = 61  # Q3.61

def parse_verilog_hex(hex_str):
    s = hex_str.lower().replace("0x", "").replace("64'h", "").replace("_", "")
    val = int(s, 16)
    if val >= 2**63:
        val -= 2**64
    return val

def q_fixed_to_float(val, frac_bits=FRAC_BITS):
    return val / float(1 << frac_bits)

def float_to_q_fixed(x, frac_bits=FRAC_BITS):
    scaled = int(round(x * (1 << frac_bits)))
    if scaled >= 2**63:
        scaled -= 2**64
    elif scaled < -(2**63):
        scaled += 2**64
    return scaled

def signed64_to_hex(val):
    if val < 0:
        val = (1 << 64) + val
    return f"64'h{val:016x}"

hex_inputs = [
     "64'hed83105a46afb400",
     "64'hfcf0c4e6380d7400",
 ]

for h in hex_inputs:
    raw = parse_verilog_hex(h)
    sign_bit = 1 if raw < 0 else 0
    x = q_fixed_to_float(raw)
    s = math.sin(x)
    c = math.cos(x)
    s_q = float_to_q_fixed(s)
    c_q = float_to_q_fixed(c)
    print(f"\nInput: {h}")
    print(f"  Sign bit: {sign_bit}")
    print(f"  Sin (float): {s}")
    print(f"  Sin (hex):   {signed64_to_hex(s_q)}")

    print(f"  Cos (float): {c}")
    print(f"  Cos (hex):   {signed64_to_hex(c_q)}")



Input: 64'hed83105a46afb400
  Sign bit: 1
  Sin (float): -0.5461413408185162
  Sin (hex):   64'hee860298461bb700
  Cos (float): 0.8376930439301459
  Cos (hex):   64'h1ace61a478888b00

Input: 64'hfcf0c4e6380d7400
  Sign bit: 1
  Sin (float): -0.09546363998289319
  Sin (hex):   64'hfcf1f63c8a1f9620
  Cos (float): 0.9954329175997831
  Cos (hex):   64'h1fda96224e7e6a00
